In [1]:
# !pip install --upgrade pip

# # Torch estable
# !pip install torch==2.1.1 --index-url https://download.pytorch.org/whl/cu121

# # Ecosistema HF estable
# !pip install --upgrade transformers
# !pip install --upgrade tokenizers
# !pip install --upgrade datasets
# !pip install --upgrade huggingface_hub
# !pip install --upgrade accelerate
# !pip install --upgrade bitsandbytes
# !pip install --upgrade peft

# # Dependencias extra
# !pip install --upgrade fsspec
# !pip install --upgrade pyarrow
# !pip install --upgrade lxml
# !pip install --upgrade sacrebleu

# !pip install -i https://test.pypi.org/simple/ lowresource-llm-evaluation==0.2.6

In [ ]:
import os
import bitsandbytes
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
from lowresource_llm_evaluation import benchmark
from lowresource_llm_evaluation.interferenciaLinguistica import loadLexicon
import pandas as pd
import numpy as np
import torch
from huggingface_hub import login
import time
import json
import gc
from dotenv import load_dotenv

base = "./"
load_dotenv(base + "secrets.env")
login(token=os.getenv("HF_TOKEN"))

2026-04-10 20:59:35.126700: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-10 20:59:35.126769: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-10 20:59:35.128311: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-10 20:59:35.137538: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-10 20:59:36.221060: W tensorflow/compiler/tf2

In [ ]:
def clean_graphics_card():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    gc.collect()
    torch.cuda.empty_cache()

def load_gallego():
    with open(base + "EvalDatasets/Raw/idioms_train_es.txt", "r", encoding="utf-8") as fEsp:
        esp = fEsp.readlines()

    with open(base + "EvalDatasets/Raw/idioms_train_gl.txt", "r", encoding="utf-8") as fGl:
        gl = fGl.readlines()

    with open(base + "EvalDatasets/Raw/idioms_test_es.txt", "r", encoding="utf-8") as fEsp:
        espTest = fEsp.readlines()

    with open(base + "EvalDatasets/Raw/idioms_test_gl.txt", "r", encoding="utf-8") as fGl:
        glTest = fGl.readlines()
    return pd.DataFrame(np.array((esp + espTest, gl + glTest)).T, columns=["es","gl"])

def split_for_translation_and_roundtrip(df, N):
    total = len(df)

    # Caso 1: hay al menos 2N → no hay solape
    if total >= 2 * N:
        df_trans = df.iloc[:N]
        df_round = df.iloc[N:2*N]
        return df_trans, df_round

    # Caso 2: no hay suficientes → roundtrip desde el final hacia atrás
    df_trans = df.iloc[:N]

    # Seleccionamos los últimos N sin tocar los primeros N
    df_round = df.iloc[-N:]

    return df_trans, df_round


def evaluate_benchmark(model_name, idioma, token, N= 20, device="cuda", debug=False, remote_code=True):
    
    # 1. Define the 4-bit quantization configuration
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=  torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    # 2. Load the tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=remote_code)

    # 3. Load the pre-trained language model with quantization
    model = AutoModelForCausalLM.from_pretrained(
                model_name,
                quantization_config=bnb_config,
                trust_remote_code=remote_code,
                tie_word_embeddings=False, # Added to silence the warning about tied weights
                token = token,
                device_map="auto"
            )
    
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id


    print(f"Tokenizer loaded: {tokenizer.__class__.__name__}")
    print(f"Model loaded with 4-bit quantization: {model.__class__.__name__}")
    print(f"Model device: {model.device}")
    codigos = {"aranes": "aran" , 
               "asturiano": "ast", 
               "gallego": "gl"}
    textos = {"aranes": pd.read_parquet("hf://datasets/projecte-aina/ES-OC_Parallel_Corpus/es-arn_corpus.parquet") , 
            "asturiano": pd.read_parquet("hf://datasets/projecte-aina/ES-AST_Parallel_Corpus/es-ast_corpus.parquet"), 
            "gallego": load_gallego()}
    
    df_textos, df_textos_round = split_for_translation_and_roundtrip(textos[idioma],N)
    del textos # Limpiamos RAM
    results = benchmark(model, tokenizer, 
            df_textos = df_textos,
            df_textos_round = df_textos_round,
            lang_eval= codigos[idioma],
            df_huecos=  pd.read_csv(base + f"EvalDatasets/Huecos/{idioma}.csv").head(N),
            df_anotado = pd.read_csv(base + f"EvalDatasets/Anotado/{idioma}.csv").head(N),
            lexicon_target = loadLexicon(base + f"lexicons/{codigos[idioma]}.txt"),
            lexicons_comparison = {"es": loadLexicon(base + f"lexicons/es.txt"), "fr": loadLexicon(base + f"lexicons/fr.txt")},
            roundtrip_langs= ["es"],
            cortar_ortografico = True,
            cortar_vocabulario = True,
            debug=debug)
    # Lberamos GPU
    try:
        model.to("cpu")
        del model
        del tokenizer
        clean_graphics_card()
    except Exception as e:
        print("Borrar el modelo ha fallado")
        print(e)
    return results


In [3]:
import torch
print(torch.cuda.is_available())
import transformers
print(transformers.__version__)

True
4.40.2


# Aranés

## Mistral 7B 

In [5]:
idioma = "aranes"
modelo = "mistralai/Mistral-7B-Instruct-v0.3"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


model-00003-of-00003.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

You are calling `save_pretrained` to a 4-bit converted model, but your `bitsandbytes` version doesn't support it. If you want to save 4-bit models, make sure to have `bitsandbytes>=0.41.3` installed.


...

TRADUCCIÓN ROUND TRIP acabado  en 21.97
Empezando VOCABULARIO


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

VOCABULARIO acabado  en 7.3
Empezando ORTOGRAFÍA


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

ORTOGRAFÍA acabado en 13.25

                        Evaluación de Calidad de Lengua                         

+-----------------+-------------------------------------------------------+
| Clave           | Valor                                                 |
+-----------------+-------------------------------------------------------+
| ttr             | 0.42874588820485016                                   |
| entropy         | 6.996158700975961                                     |
| ngram_overlap   | 0.0                                                   |
| freq_target     | 0.4750648949422809                                    |
| freq_comparison | {'es': 0.5071562368146278, 'fr': 0.33025804903248845} |
| calidad         | 0.14272736102783534                                   |
+-----------------+-------------------------------------------------------+

                            Evaluación de Traducción                            

+------+--------------------+
| Clave | Valor 

## Salamandra

In [6]:
idioma = "aranes"
modelo = "BSC-LT/salamandra-7b-instruct"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.81M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/19.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/513 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/730 [00:00<?, ?B/s]

...

TRADUCCIÓN ROUND TRIP acabado  en 5.64
Empezando VOCABULARIO


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

VOCABULARIO acabado  en 1.22
Empezando ORTOGRAFÍA


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

ORTOGRAFÍA acabado en 4.86

                        Evaluación de Calidad de Lengua                         

+-----------------+-------------------------------------------------------+
| Clave           | Valor                                                 |
+-----------------+-------------------------------------------------------+
| ttr             | 0.7929654126530111                                    |
| entropy         | 6.974806547089256                                     |
| ngram_overlap   | 0.0015503875968992248                                 |
| freq_target     | 0.347543409649552                                     |
| freq_comparison | {'es': 0.37589956340959063, 'fr': 0.4633192219840657} |
| calidad         | 0.22821040572173484                                   |
+-----------------+-------------------------------------------------------+

                            Evaluación de Traducción                            

+------+-------------------+
| Clave | Valor   

## Gemma

In [7]:
idioma = "aranes"
modelo = "google/gemma-7b-it"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

You are calling `save_pretrained` to a 4-bit converted model, but your `bitsandbytes` version doesn't support it. If you want to save 4-bit models, make sure to have `bitsandbytes>=0.41.3` installed.


Tokenizer loaded: GemmaTokenizerFast
Model loaded with 4-bit quantization: GemmaForCausalLM
Model device: cuda:0
Empezando CALIDAD DE LENGUA
CALIDAD DE LENGUA acabada en 5.61 minutos
Empezando TRADUCCIÓN DIRECTA
TRADUCCIÓN DIRECTA acabada en 14.73
Empezando TRADUCCIÓN ROUND TRIP
TRADUCCIÓN ROUND TRIP acabado  en 29.95
Empezando VOCABULARIO
VOCABULARIO acabado  en 7.73
Empezando ORTOGRAFÍA


You shouldn't move a model when it is dispatched on multiple devices.


ORTOGRAFÍA acabado en 14.84

                        Evaluación de Calidad de Lengua                         

+-----------------+---------------------------------------------------------+
| Clave           | Valor                                                   |
+-----------------+---------------------------------------------------------+
| ttr             | 0.6776458212393024                                      |
| entropy         | 6.478566334131271                                       |
| ngram_overlap   | 0.0                                                     |
| freq_target     | 0.007557516253798244                                    |
| freq_comparison | {'es': 0.005759257054118212, 'fr': 0.12129867600726689} |
| calidad         | 0.2239458878853342                                      |
+-----------------+---------------------------------------------------------+

                            Evaluación de Traducción                            

+------+------------------

## Qwen

In [8]:
idioma = "aranes"
modelo = "Qwen/Qwen2.5-7B-Instruct"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


...

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

You are calling `save_pretrained` to a 4-bit converted model, but your `bitsandbytes` version doesn't support it. If you want to save 4-bit models, make sure to have `bitsandbytes>=0.41.3` installed.


Tokenizer loaded: Qwen2TokenizerFast
Model loaded with 4-bit quantization: Qwen2ForCausalLM
Model device: cuda:0
Empezando CALIDAD DE LENGUA
CALIDAD DE LENGUA acabada en 4.81 minutos
Empezando TRADUCCIÓN DIRECTA
TRADUCCIÓN DIRECTA acabada en 6.36
Empezando TRADUCCIÓN ROUND TRIP
TRADUCCIÓN ROUND TRIP acabado  en 21.1
Empezando VOCABULARIO
VOCABULARIO acabado  en 7.2
Empezando ORTOGRAFÍA


You shouldn't move a model when it is dispatched on multiple devices.


ORTOGRAFÍA acabado en 14.21

                        Evaluación de Calidad de Lengua                         

+-----------------+-------------------------------------------------------+
| Clave           | Valor                                                 |
+-----------------+-------------------------------------------------------+
| ttr             | 0.33463123738192446                                   |
| entropy         | 6.4720381347804175                                    |
| ngram_overlap   | 0.0                                                   |
| freq_target     | 0.49453991620019727                                   |
| freq_comparison | {'es': 0.5285321504940859, 'fr': 0.37326925267105976} |
| calidad         | 0.0985598478102161                                    |
+-----------------+-------------------------------------------------------+

                            Evaluación de Traducción                            

+------+-------------------+
| Clave | Valor  

# Asturiano

## Mistral 7B 

In [9]:
idioma = "asturiano"
modelo = "mistralai/Mistral-7B-Instruct-v0.3"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

You are calling `save_pretrained` to a 4-bit converted model, but your `bitsandbytes` version doesn't support it. If you want to save 4-bit models, make sure to have `bitsandbytes>=0.41.3` installed.


Tokenizer loaded: LlamaTokenizerFast
Model loaded with 4-bit quantization: MistralForCausalLM
Model device: cuda:0


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Empezando CALIDAD DE LENGUA


...

TRADUCCIÓN ROUND TRIP acabado  en 21.64
Empezando VOCABULARIO


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

VOCABULARIO acabado  en 7.19
Empezando ORTOGRAFÍA


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

ORTOGRAFÍA acabado en 10.92

                        Evaluación de Calidad de Lengua                         

+-----------------+------------------------------------------------------+
| Clave           | Valor                                                |
+-----------------+------------------------------------------------------+
| ttr             | 0.46407879234109545                                  |
| entropy         | 7.0478943378224725                                   |
| ngram_overlap   | 0.0                                                  |
| freq_target     | 0.5253611865382336                                   |
| freq_comparison | {'es': 0.769985928545387, 'fr': 0.36414306594952295} |
| calidad         | 0.14651293798555798                                  |
+-----------------+------------------------------------------------------+

                            Evaluación de Traducción                            

+------+--------------------+
| Clave | Valor           

## Salamandra

In [10]:
idioma = "asturiano"
modelo = "BSC-LT/salamandra-7b-instruct"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

You are calling `save_pretrained` to a 4-bit converted model, but your `bitsandbytes` version doesn't support it. If you want to save 4-bit models, make sure to have `bitsandbytes>=0.41.3` installed.


Tokenizer loaded: LlamaTokenizerFast
Model loaded with 4-bit quantization: LlamaForCausalLM
Model device: cuda:0


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Empezando CALIDAD DE LENGUA


...

TRADUCCIÓN ROUND TRIP acabado  en 27.41
Empezando VOCABULARIO


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

VOCABULARIO acabado  en 4.11
Empezando ORTOGRAFÍA


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

ORTOGRAFÍA acabado en 13.98

                        Evaluación de Calidad de Lengua                         

+-----------------+-------------------------------------------------------+
| Clave           | Valor                                                 |
+-----------------+-------------------------------------------------------+
| ttr             | 0.8281316569730588                                    |
| entropy         | 8.127123470132833                                     |
| ngram_overlap   | 0.0                                                   |
| freq_target     | 0.46927934046800057                                   |
| freq_comparison | {'es': 0.7552048514192844, 'fr': 0.23957284474051493} |
| calidad         | 0.17088073840600354                                   |
+-----------------+-------------------------------------------------------+

                            Evaluación de Traducción                            

+------+--------------------+
| Clave | Valor 

## Gemma

In [4]:
idioma = "asturiano"
modelo = "google/gemma-7b-it"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

You are calling `save_pretrained` to a 4-bit converted model, but your `bitsandbytes` version doesn't support it. If you want to save 4-bit models, make sure to have `bitsandbytes>=0.41.3` installed.


Tokenizer loaded: GemmaTokenizerFast
Model loaded with 4-bit quantization: GemmaForCausalLM
Model device: cuda:0
Empezando CALIDAD DE LENGUA
CALIDAD DE LENGUA acabada en 6.19 minutos
Empezando TRADUCCIÓN DIRECTA
TRADUCCIÓN DIRECTA acabada en 16.12
Empezando TRADUCCIÓN ROUND TRIP
TRADUCCIÓN ROUND TRIP acabado  en 32.45
Empezando VOCABULARIO
VOCABULARIO acabado  en 8.4
Empezando ORTOGRAFÍA


You shouldn't move a model when it is dispatched on multiple devices.


ORTOGRAFÍA acabado en 16.12

                        Evaluación de Calidad de Lengua                         

+-----------------+---------------------------------------------------------+
| Clave           | Valor                                                   |
+-----------------+---------------------------------------------------------+
| ttr             | 0.9339296697252777                                      |
| entropy         | 6.032629898353748                                       |
| ngram_overlap   | 0.0                                                     |
| freq_target     | 0.010957657599448644                                    |
| freq_comparison | {'es': 0.07361248412986512, 'fr': 0.027342176442227147} |
| calidad         | 0.23387195017668216                                     |
+-----------------+---------------------------------------------------------+

                            Evaluación de Traducción                            

+------+------------------

## Qwen

In [5]:
idioma = "asturiano"
modelo = "Qwen/Qwen2.5-7B-Instruct"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


...

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

You are calling `save_pretrained` to a 4-bit converted model, but your `bitsandbytes` version doesn't support it. If you want to save 4-bit models, make sure to have `bitsandbytes>=0.41.3` installed.


Tokenizer loaded: Qwen2TokenizerFast
Model loaded with 4-bit quantization: Qwen2ForCausalLM
Model device: cuda:0
Empezando CALIDAD DE LENGUA
CALIDAD DE LENGUA acabada en 3.47 minutos
Empezando TRADUCCIÓN DIRECTA
TRADUCCIÓN DIRECTA acabada en 10.09
Empezando TRADUCCIÓN ROUND TRIP
TRADUCCIÓN ROUND TRIP acabado  en 20.39
Empezando VOCABULARIO
VOCABULARIO acabado  en 6.1
Empezando ORTOGRAFÍA


You shouldn't move a model when it is dispatched on multiple devices.


ORTOGRAFÍA acabado en 9.77

                        Evaluación de Calidad de Lengua                         

+-----------------+------------------------------------------------------+
| Clave           | Valor                                                |
+-----------------+------------------------------------------------------+
| ttr             | 0.39654980047688176                                  |
| entropy         | 6.441992120151433                                    |
| ngram_overlap   | 0.0011111111111111111                                |
| freq_target     | 0.6139667290902597                                   |
| freq_comparison | {'es': 0.8982316948380878, 'fr': 0.3706671004659364} |
| calidad         | 0.1309384954264295                                   |
+-----------------+------------------------------------------------------+

                            Evaluación de Traducción                            

+------+-------------------+
| Clave | Valor             

# Gallego

## Mistral

In [6]:
idioma = "gallego"
modelo = "mistralai/Mistral-7B-Instruct-v0.3"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


...

TRADUCCIÓN ROUND TRIP acabado  en 20.18
Empezando VOCABULARIO


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

VOCABULARIO acabado  en 5.43
Empezando ORTOGRAFÍA


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

ORTOGRAFÍA acabado en 13.53

                        Evaluación de Calidad de Lengua                         

+-----------------+-------------------------------------------------------+
| Clave           | Valor                                                 |
+-----------------+-------------------------------------------------------+
| ttr             | 0.5060249996402517                                    |
| entropy         | 6.818829987962138                                     |
| ngram_overlap   | 0.0005865102639296188                                 |
| freq_target     | 0.630740117431382                                     |
| freq_comparison | {'es': 0.5564961816408701, 'fr': 0.32357230061049475} |
| calidad         | 0.2438703416099588                                    |
+-----------------+-------------------------------------------------------+

                            Evaluación de Traducción                            

+------+--------------------+
| Clave | Valor 

## Salamandra

In [7]:
idioma = "gallego"
modelo = "BSC-LT/salamandra-7b-instruct"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.81M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/19.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/513 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/730 [00:00<?, ?B/s]

...

TRADUCCIÓN ROUND TRIP acabado  en 30.44
Empezando VOCABULARIO


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

VOCABULARIO acabado  en 9.15
Empezando ORTOGRAFÍA


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

ORTOGRAFÍA acabado en 15.2

                        Evaluación de Calidad de Lengua                         

+-----------------+-------------------------------------------------------+
| Clave           | Valor                                                 |
+-----------------+-------------------------------------------------------+
| ttr             | 0.7529228107872835                                    |
| entropy         | 7.7675978212138705                                    |
| ngram_overlap   | 0.0003179650238473768                                 |
| freq_target     | 0.8509263229980839                                    |
| freq_comparison | {'es': 0.5681900527932997, 'fr': 0.24495914434182192} |
| calidad         | 0.38377521905603135                                   |
+-----------------+-------------------------------------------------------+

                            Evaluación de Traducción                            

+------+--------------------+
| Clave | Valor  

## Gemma

In [4]:
idioma = "gallego"
modelo = "google/gemma-7b-it"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

You are calling `save_pretrained` to a 4-bit converted model, but your `bitsandbytes` version doesn't support it. If you want to save 4-bit models, make sure to have `bitsandbytes>=0.41.3` installed.


Tokenizer loaded: GemmaTokenizerFast
Model loaded with 4-bit quantization: GemmaForCausalLM
Model device: cuda:0
Empezando CALIDAD DE LENGUA
CALIDAD DE LENGUA acabada en 5.55 minutos
Empezando TRADUCCIÓN DIRECTA
TRADUCCIÓN DIRECTA acabada en 14.66
Empezando TRADUCCIÓN ROUND TRIP
TRADUCCIÓN ROUND TRIP acabado  en 29.83
Empezando VOCABULARIO
VOCABULARIO acabado  en 7.61
Empezando ORTOGRAFÍA


You shouldn't move a model when it is dispatched on multiple devices.


ORTOGRAFÍA acabado en 14.7

                        Evaluación de Calidad de Lengua                         

+-----------------+-----------------------------------------------------------+
| Clave           | Valor                                                     |
+-----------------+-----------------------------------------------------------+
| ttr             | 0.5355556901902265                                        |
| entropy         | 6.475961285204481                                         |
| ngram_overlap   | 0.0                                                       |
| freq_target     | 0.00864291482985377                                       |
| freq_comparison | {'fr': 0.0024761904761904764, 'es': 0.011923078429035776} |
| calidad         | 0.1741203705913071                                        |
+-----------------+-----------------------------------------------------------+

                            Evaluación de Traducción                            

+------

## Qwen

In [5]:
idioma = "gallego"
modelo = "Qwen/Qwen2.5-7B-Instruct"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


...

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

You are calling `save_pretrained` to a 4-bit converted model, but your `bitsandbytes` version doesn't support it. If you want to save 4-bit models, make sure to have `bitsandbytes>=0.41.3` installed.


Tokenizer loaded: Qwen2TokenizerFast
Model loaded with 4-bit quantization: Qwen2ForCausalLM
Model device: cuda:0
Empezando CALIDAD DE LENGUA
CALIDAD DE LENGUA acabada en 4.31 minutos
Empezando TRADUCCIÓN DIRECTA
TRADUCCIÓN DIRECTA acabada en 6.77
Empezando TRADUCCIÓN ROUND TRIP
TRADUCCIÓN ROUND TRIP acabado  en 18.53
Empezando VOCABULARIO
VOCABULARIO acabado  en 3.98
Empezando ORTOGRAFÍA


You shouldn't move a model when it is dispatched on multiple devices.


ORTOGRAFÍA acabado en 5.68

                        Evaluación de Calidad de Lengua                         

+-----------------+------------------------------------------------------+
| Clave           | Valor                                                |
+-----------------+------------------------------------------------------+
| ttr             | 0.390470673306703                                    |
| entropy         | 6.294283895252525                                    |
| ngram_overlap   | 0.0037260456509126015                                |
| freq_target     | 0.8742289668690362                                   |
| freq_comparison | {'fr': 0.3528625115896761, 'es': 0.6804082047205073} |
| calidad         | 0.2261275381729694                                   |
+-----------------+------------------------------------------------------+

                            Evaluación de Traducción                            

+------+--------------------+
| Clave | Valor            